# 🤟 Wasel v4 Pro — Gemini AI Live Translator

1. شغّل **Cell 1** — تثبيت
2. شغّل **Cell 2** — الصق مفتاح Gemini → الكاميرا تبدأ تلقائياً!

In [ ]:
#@title 📦 Cell 1: Install
!pip install -q "google-genai>=1.0.0" "flask" "flask-cors" "Pillow"
print("✅ Ready! Run Cell 2.")

In [ ]:
#@title 🎥 Cell 2: Launch Live Translator
#@markdown Gemini API Key (free from aistudio.google.com/apikey):
GEMINI_API_KEY = "" #@param {type:"string"}

import threading, time, base64, io
from flask import Flask, request, jsonify, Response
from flask_cors import CORS
from PIL import Image
import google.genai as genai
from google.genai import types

if not GEMINI_API_KEY:
    raise ValueError("❌ Paste Gemini API key!")

client = genai.Client(api_key=GEMINI_API_KEY)
MODEL = "gemini-2.0-flash"
test = client.models.generate_content(model=MODEL, contents="Say OK")
print(f"✅ Gemini: {test.text.strip()}")

PROMPT = """You are a sign language interpreter.
If you see a hand gesture or sign: reply ONLY the meaning (1-3 words).
If no gesture: reply ...
No explanations. Just the word."""

app = Flask(__name__)
CORS(app)

PAGE = r"""
<!DOCTYPE html><html><head><meta charset='utf-8'>
<meta name='viewport' content='width=device-width,initial-scale=1'>
<title>Wasel v4 Pro</title>
<style>
*{margin:0;padding:0;box-sizing:border-box}
body{background:#0a0a0a;font-family:'Segoe UI',sans-serif;overflow:hidden;height:100vh}
#c{position:relative;width:100vw;height:100vh}
video{width:100%;height:100%;object-fit:cover;transform:scaleX(-1)}
#top{position:absolute;top:0;left:0;right:0;padding:18px 28px;
background:linear-gradient(180deg,rgba(0,0,0,0.85) 0%,transparent 100%)}
#brand{color:#666;font-size:13px;letter-spacing:3px;text-transform:uppercase}
#txt{color:#00ff88;font-size:42px;font-weight:700;margin-top:6px;
text-shadow:0 2px 20px rgba(0,255,136,0.4);min-height:55px;transition:all .3s}
#bot{position:absolute;bottom:16px;right:24px;color:#444;font-size:12px}
</style></head><body>
<div id='c'>
<video id='v' autoplay playsinline muted></video>
<div id='top'><div id='brand'>WASEL v4 PRO — GEMINI AI</div>
<div id='txt'>Starting camera...</div></div>
<div id='bot'>Gemini 2.0 Flash</div>
</div>
<canvas id='cv' style='display:none'></canvas>
<script>
const v=document.getElementById('v'),cv=document.getElementById('cv'),
cx=cv.getContext('2d'),tx=document.getElementById('txt'),
bt=document.getElementById('bot');
let busy=false;
navigator.mediaDevices.getUserMedia({video:{width:640,height:480,facingMode:'user'}})
.then(s=>{v.srcObject=s;tx.textContent='Show a sign...';tx.style.color='#555';
setInterval(go,2000)}).catch(e=>{tx.textContent='Camera: '+e.message});
function go(){if(busy)return;busy=true;cv.width=512;cv.height=384;
cx.drawImage(v,0,0,512,384);
const d=cv.toDataURL('image/jpeg',0.6);
bt.textContent='⚡ Analyzing...';
fetch('/translate',{method:'POST',headers:{'Content-Type':'application/json'},
body:JSON.stringify({image:d})})
.then(r=>r.json()).then(d=>{
const t=d.translation||'...';
if(t==='...'||t.length<2){tx.textContent='Show a sign...';tx.style.color='#555'}
else{tx.textContent=t;tx.style.color='#00ff88'}
bt.textContent='⚡ Gemini 2.0 — '+new Date().toLocaleTimeString();busy=false
}).catch(e=>{bt.textContent='Err: '+e;busy=false})}
</script></body></html>
"""

@app.route('/')
def index():
    return Response(PAGE, mimetype='text/html')

@app.route('/translate', methods=['POST'])
def translate():
    try:
        d = request.json
        img_b = base64.b64decode(d['image'].split(',')[1])
        pil = Image.open(io.BytesIO(img_b))
        r = client.models.generate_content(
            model=MODEL, contents=[PROMPT, pil],
            config=types.GenerateContentConfig(max_output_tokens=20, temperature=0.1)
        )
        return jsonify({'translation': r.text.strip()})
    except Exception as e:
        return jsonify({'translation': f'Error: {str(e)[:40]}'})

# ─── Start ───
threading.Thread(target=lambda: app.run(port=5000, debug=False, use_reloader=False), daemon=True).start()
time.sleep(2)

# Public URL via Colab proxy (no ngrok needed!)
from google.colab import output
output.serve_kernel_port_as_window(5000)

print("")
print("══════════════════════════════════════")
print("  🤟 Wasel v4 Pro — LIVE!")
print("══════════════════════════════════════")
print("")
print("  نافذة جديدة ستفتح تلقائياً!")
print("  إذا لم تفتح، اضغط الرابط أعلاه.")
print("")
print("  📸 اسمح للكاميرا → أعمل إشارة → الترجمة فوراً!")
print("══════════════════════════════════════")